# Home Credit: data cleaning and preprocessing
We will write the code together, one section at a time. Each section contains an explanation and an empty practice cell.

**Workflow:** load and verify; investigate types, missingness, categories, ranges, and outliers; document and apply confirmed rules; split; fit preprocessing on training rows only; verify and save.

Our data-understanding notebook is the reference for earlier findings. This notebook runs independently and reads application_train.csv again. The Kaggle application_test.csv has no TARGET and is not our labeled validation set.

Inspection can suggest a rule, but we must distinguish fixed, justified corrections from thresholds or statistics learned from training data. Document what changed and why. Keep the original CSVs unchanged.

**Current scope: application_train.csv only.** Sections 11-22 finish cleaning, feature engineering, encoding, missing-value handling, optional scaling, verification, and a reproducible train/validation handoff. The source labeled CSV is split to protect validation; both portions receive consistent preparation. Kaggle test and historical CSV processing will be separate later work. Your teammate handles model training and evaluation. New practice cells stay empty for us to implement together.


## 1. Load the original training data
Import Path from pathlib and import pandas using the alias pd. Set data_folder to home-credit-default-risk, then read application_train.csv into df.

**Your exercise:** display df.shape and the first five rows using head(). Expected size from our earlier inspection: 307,511 rows and 122 columns. If the file is not found, check the working directory before changing the path.

**Why:** this notebook needs its own imports and data loading so it can run independently. Read the original CSV without overwriting it.


In [2]:
from pathlib import Path
import pandas as pd
# Locate the project from either ProJect/ or ProJect/notebooks/.
project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "data" / "raw" / "home-credit-default-risk" / "application_train.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from inside the ProJect folder.")
folder = project_root / "data" / "raw" / "home-credit-default-risk"
df = pd.read_csv(folder / "application_train.csv")

display(df.head())
display(df.shape)



,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


(307511, 122)

## 2. Verify application IDs and target values
Count duplicate complete rows, repeated SK_ID_CURR values, missing IDs, missing TARGET values, and nonmissing TARGET values outside 0 and 1. Display target counts and percentages.

**Why:** IDs keep records aligned; supervised learning requires valid recorded outcomes. If checks fail, investigate before deleting or changing rows. Never fill missing TARGET using a feature-imputation strategy.

**Practice:** print a labeled result for each check. Record any issue that prevents splitting.


In [5]:
duplicated_rows = df.duplicated().sum()
print("Number of duplicated rows:", duplicated_rows)

duplicated_Id = df["SK_ID_CURR"].duplicated().sum()
print("Number of duplicated SK_ID_CURR:", duplicated_Id)

isnull_id = df["SK_ID_CURR"].isnull().sum()
print("Number of missing SK_ID_CURR:", isnull_id)

isnull_target = df["TARGET"].isnull().sum()
print("Number of missing TARGET:", isnull_target)

target_counts = df["TARGET"].value_counts(dropna=False)
display(target_counts)

target_counts_percent = df["TARGET"].value_counts(normalize=True) * 100
display(target_counts_percent)


unexpected_values = df["TARGET"].unique()
display(unexpected_values)

Number of duplicated rows: 0
Number of duplicated SK_ID_CURR: 0
Number of missing SK_ID_CURR: 0
Number of missing TARGET: 0


TARGET
0    282686
1     24825
Name: count, dtype: int64

TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64

array([1, 0])

## 3. Create a working copy and a decision log
Create df_clean using df.copy(). Confirm the same starting shape and retain df as the original reference.

Create a Markdown cell for cleaning decisions with columns: feature, issue, evidence, action, reason, affected rows, and verification. Start with pending decisions; update them as we implement rules.

**Why:** cleaning should be explainable. We need to distinguish a proposed rule from an applied change.


In [7]:
df_clean = df.copy()

display(df.shape)
display(df_clean.shape)

(307511, 122)

(307511, 122)

### Cleaning decision log
This table records proposed and completed decisions. Pending means the action has not been applied. Update evidence, counts, and verification as we work.

| Feature | Issue | Evidence | Action | Reason | Affected rows | Verification |
|---|---|---|---|---|---|---|
| DAYS_EMPLOYED | Placeholder 365243 | 55,374 source records have the implausible employment duration | Applied: add DAYS_EMPLOYED_PLACEHOLDER and replace 365243 with numeric missing | Preserve applications and record unavailable duration | 55,374 | Verified on two consecutive executions: 307,511 rows; 0 placeholders remain; 55,374 flags; 55,374 missing employment values; original df unchanged |
| AMT_INCOME_TOTAL | Extreme income | Maximum recorded income of 117,000,000 in data understanding | Pending review; no change applied | A high value alone does not establish a data error | Not yet counted under a defined rule | Pending |


## 4. Validate data types and column meanings
Review stored types and check their meanings against the column dictionary. Confirm financial amounts are numeric, text categories are treated as categories, and IDs remain identifiers.

**Important:** DAYS_BIRTH and DAYS_EMPLOYED are relative day offsets, not calendar dates. Negative offsets may be expected. Do not run calendar-date conversion on them or treat every numeric column as a continuous measurement.

**Practice:** inspect types and document actual mismatches. Convert only confirmed mismatches, recording whether conversion introduces missing values. Do not force all columns to the same type.


In [11]:
df_clean.info(verbose=True, show_counts=True)



columns_to_check = [
    "SK_ID_CURR",
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "NAME_CONTRACT_TYPE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
]


df_clean[columns_to_check].dtypes

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 122 columns):
 #    Column                        Non-Null Count   Dtype  
---   ------                        --------------   -----  
 0    SK_ID_CURR                    307511 non-null  int64  
 1    TARGET                        307511 non-null  int64  
 2    NAME_CONTRACT_TYPE            307511 non-null  str    
 3    CODE_GENDER                   307511 non-null  str    
 4    FLAG_OWN_CAR                  307511 non-null  str    
 5    FLAG_OWN_REALTY               307511 non-null  str    
 6    CNT_CHILDREN                  307511 non-null  int64  
 7    AMT_INCOME_TOTAL              307511 non-null  float64
 8    AMT_CREDIT                    307511 non-null  float64
 9    AMT_ANNUITY                   307499 non-null  float64
 10   AMT_GOODS_PRICE               307233 non-null  float64
 11   NAME_TYPE_SUITE               306219 non-null  str    
 12   NAME_INCOME_TYPE              307511 no

SK_ID_CURR              int64
TARGET                  int64
AMT_INCOME_TOTAL      float64
AMT_CREDIT            float64
NAME_CONTRACT_TYPE        str
DAYS_BIRTH              int64
DAYS_EMPLOYED           int64
dtype: object

## 5. Investigate missing percentages and patterns
Calculate missing counts and percentages for df_clean. Display the largest percentages and inspect related columns to explore possible reasons.

**Example:** compare missing OWN_CAR_AGE with FLAG_OWN_CAR. Missing car age for someone without a car may have a different meaning from missing age for a car owner. Also inspect whether groups of housing columns are missing together.

**Practice:** build a small missing-value report and record hypotheses, keeping observations separate from assumptions. Do not fill values or automatically remove columns yet. Any data-derived missingness threshold must be chosen using training rows after splitting.


In [23]:
missing_counts = df_clean.isna().sum()
print("Missing values per column:")
display(missing_counts)

missing_percentages = (missing_counts / len(df_clean)) * 100
print("Missing values percentage per column:")
display(missing_percentages)


missing_report = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percentages,
})

display(
    missing_report
    .sort_values("missing_percent", ascending=False)
    .head(15)
    .round(2)
)




print(
    "Columns with at least one missing value:",
    (missing_report["missing_count"] > 0).sum(),
)

print(
    "Columns with at least 20% missing:",
    (missing_report["missing_percent"] >= 20).sum(),
)

print(
    "Columns entirely missing:",
    (missing_report["missing_percent"] == 100).sum(),
)






Missing values per column:


SK_ID_CURR                        0
TARGET                            0
NAME_CONTRACT_TYPE                0
CODE_GENDER                       0
FLAG_OWN_CAR                      0
                              ...  
AMT_REQ_CREDIT_BUREAU_DAY     41519
AMT_REQ_CREDIT_BUREAU_WEEK    41519
AMT_REQ_CREDIT_BUREAU_MON     41519
AMT_REQ_CREDIT_BUREAU_QRT     41519
AMT_REQ_CREDIT_BUREAU_YEAR    41519
Length: 122, dtype: int64

Missing values percentage per column:


SK_ID_CURR                     0.000000
TARGET                         0.000000
NAME_CONTRACT_TYPE             0.000000
CODE_GENDER                    0.000000
FLAG_OWN_CAR                   0.000000
                                ...    
AMT_REQ_CREDIT_BUREAU_DAY     13.501631
AMT_REQ_CREDIT_BUREAU_WEEK    13.501631
AMT_REQ_CREDIT_BUREAU_MON     13.501631
AMT_REQ_CREDIT_BUREAU_QRT     13.501631
AMT_REQ_CREDIT_BUREAU_YEAR    13.501631
Length: 122, dtype: float64

,missing_count,missing_percent
COMMONAREA_AVG,214865,69.87
COMMONAREA_MODE,214865,69.87
COMMONAREA_MEDI,214865,69.87
NONLIVINGAPARTMENTS_MEDI,213514,69.43
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_AVG,213514,69.43
FONDKAPREMONT_MODE,210295,68.39
LIVINGAPARTMENTS_AVG,210199,68.35
LIVINGAPARTMENTS_MEDI,210199,68.35
LIVINGAPARTMENTS_MODE,210199,68.35


Columns with at least one missing value: 67
Columns with at least 20% missing: 50
Columns entirely missing: 0


In [ ]:

car_check = df_clean[["FLAG_OWN_CAR", "OWN_CAR_AGE"]].copy()

car_check["car_age_missing"] = car_check["OWN_CAR_AGE"].isna()
display(car_check.head(10))







car_missing_report = (
    car_check.groupby("FLAG_OWN_CAR", dropna=False)
    .agg(
        applications=("car_age_missing", "size"),
        missing_car_age=("car_age_missing", "sum"),
    )
)

car_missing_report["missing_percent"] = (
    car_missing_report["missing_car_age"]
    / car_missing_report["applications"]
    * 100
)

display(car_missing_report.round(5))


In [25]:
housing_columns = [
    "APARTMENTS_AVG",
    "LIVINGAREA_AVG",
    "COMMONAREA_AVG",
]

housing_missing = df_clean[housing_columns].isna()

housing_patterns = housing_missing.value_counts(
    dropna=False
).rename("applications")

display(housing_patterns)

APARTMENTS_AVG  LIVINGAREA_AVG  COMMONAREA_AVG
True            True            True              150185
False           False           False              92095
                                True               55193
True            False           True                5398
False           True            True                4089
True            False           False                475
False           True            False                 73
True            True            False                  3
Name: applications, dtype: int64

### Proposed missing-value decisions
These are proposed strategies, not applied changes. We will finish the categorical and numerical investigations, split the labeled data, and fit replacement values using training rows only. Validation and future applications will use the same fitted replacements.

| Feature or group | Proposed action | Reason | Status |
|---|---|---|---|
| Numeric features such as AMT_ANNUITY | Fill missing values with the training median | Provides a typical value less influenced by extremes than the mean | Pending; calculate after splitting |
| Text features such as OCCUPATION_TYPE | Fill missing entries with an explicit Missing label before encoding | Preserves the fact that information was unavailable | Pending; implement in preprocessing |
| DAYS_EMPLOYED | Flag the exact placeholder 365243, replace it with numeric missing, and later impute from training data | The placeholder is not a literal employment duration | Pending confirmed cleaning rule |
| OWN_CAR_AGE | Retain FLAG_OWN_CAR, add an age-missing indicator, and later fill the numeric gap using the training median | Ownership distinguishes no car from a car with unrecorded age; the flag identifies imputed ages | Proposed; interpret section 5 group results before finalizing |
| Features with large missing percentages | Review meaning and training-set missingness before deciding whether to retain or exclude | High missingness alone does not establish that a feature is useless | Pending training-set review |
| Features entirely missing in training | Explicitly decide whether to exclude or retain with a defined fallback | A median cannot be estimated when no training values are recorded | Pending training-set review |

**Example:** recorded training annuities of 10,000, 20,000, and 90,000 have median 20,000. That value would fill missing annuities in both training and validation; validation does not get a separately calculated median.

**Next:** investigate categorical values in section 6. Do not fill values yet or replace every missing entry with zero. Update the main decision log when a rule is finalized and applied.


### What step 5 changes, and when cleaning happens
**Step 5 is inspection only.** The missing reports, car_check, and housing_missing are separate summaries or temporary inspection tables. No values in df_clean have been filled, removed, or corrected here. The car-age indicator currently exists only in car_check, not in our model inputs.

**Findings from the saved inspection:** 67 columns have missing data, 50 have at least 20% missing, and none is entirely missing. All 202,924 nonowners lack car age; only 5 of 104,587 owners lack it (approximately 0.00478%). The three inspected housing fields are all missing together in 150,185 applications. These findings support investigation and proposed decisions; they do not automatically justify deleting features.

**Our next actions:**
1. Sections 6-8 inspect categories, impossible ranges, and outliers so we can justify changes.
2. Section 9 applies confirmed corrections, starting with flagging employment value 365243 and replacing it with missing. Record every applied rule in the decision log.
3. Section 10 removes TARGET and SK_ID_CURR from model inputs (retaining outcomes and IDs separately) and splits training from validation.
4. Section 11 decides whether to exclude any features using training-set evidence. A column is not unused merely because it has missing entries.
5. Sections 12-13 define and apply missing-value filling and encoding using training-fitted rules. Those are the steps that prepare complete numeric model inputs.

For our first version we propose retaining useful partially observed features, using training medians for numeric gaps and an explicit Missing category for text. Any feature exclusions must be documented and applied consistently to validation and future inputs.

**Display adjustment:** the detailed car report uses five decimal places because two decimals incorrectly looks like zero for the five car owners with missing age. Run the car-report cell again after this edit. No CSV was changed.


## 6. Investigate categorical values
Inspect category counts with missing entries included for NAME_INCOME_TYPE, NAME_CONTRACT_TYPE, CODE_GENDER, and ORGANIZATION_TYPE. Look for placeholders such as XNA or Unknown, unexpected labels, and leading or trailing whitespace.

**Practice:** check the dictionary and context before deciding whether a label is valid, unknown, or inconsistent. Record proposed mappings and their counts. A label that looks unusual is not automatically an error.

Normalize whitespace or spelling only when the rule is justified. Learned rare-category grouping must be fitted on training data only.


In [27]:
categorical_columns = [
    "NAME_INCOME_TYPE",
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "ORGANIZATION_TYPE",
]

for column in categorical_columns:
    print(f"\nColumn: {column}")
    display(df_clean[column].value_counts(dropna=False))






Column: NAME_INCOME_TYPE


NAME_INCOME_TYPE
Working                 158774
Commercial associate     71617
Pensioner                55362
State servant            21703
Unemployed                  22
Student                     18
Businessman                 10
Maternity leave              5
Name: count, dtype: int64


Column: NAME_CONTRACT_TYPE


NAME_CONTRACT_TYPE
Cash loans         278232
Revolving loans     29279
Name: count, dtype: int64


Column: CODE_GENDER


CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64


Column: ORGANIZATION_TYPE


ORGANIZATION_TYPE
Business Entity Type 3    67992
XNA                       55374
Self-employed             38412
Other                     16683
Medicine                  11193
Business Entity Type 2    10553
Government                10404
School                     8893
Trade: type 7              7831
Kindergarten               6880
Construction               6721
Business Entity Type 1     5984
Transport: type 4          5398
Trade: type 3              3492
Industry: type 9           3368
Industry: type 3           3278
Security                   3247
Housing                    2958
Industry: type 11          2704
Military                   2634
Bank                       2507
Agriculture                2454
Police                     2341
Transport: type 2          2204
Postal                     2157
Security Ministries        1974
Trade: type 2              1900
Restaurant                 1811
Services                   1575
University                 1327
Industry: type 7      

In [ ]:
organization_unspecified = df_clean["ORGANIZATION_TYPE"] == "XNA"

display(
    df_clean.loc[
        organization_unspecified,
        "NAME_INCOME_TYPE",
    ].value_counts(dropna=False)
)




NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64

In [29]:
text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    print(f"\nColumn: {column}")
    display(df_clean[column].value_counts(dropna=False))


Column: NAME_CONTRACT_TYPE


NAME_CONTRACT_TYPE
Cash loans         278232
Revolving loans     29279
Name: count, dtype: int64


Column: CODE_GENDER


CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64


Column: FLAG_OWN_CAR


FLAG_OWN_CAR
N    202924
Y    104587
Name: count, dtype: int64

... (22 outputs left out)


Column: WALLSMATERIAL_MODE


WALLSMATERIAL_MODE
NaN             156341
Panel            66040
Stone, brick     64815
Block             9253
Wooden            5362
Mixed             2296
Monolithic        1779
Others            1625
Name: count, dtype: int64


Column: EMERGENCYSTATE_MODE


EMERGENCYSTATE_MODE
No     159428
NaN    145755
Yes      2328
Name: count, dtype: int64

## 7. Validate numeric ranges and date offsets
Check relevant financial amounts for nonpositive values and interpret results separately for each column. Zero may be valid in some financial fields. Estimate age from -DAYS_BIRTH / 365.25 for review and inspect the observed range.

**Practice:** count suspicious values, display example records, and document which are confirmed invalid versus still under investigation. Choose age plausibility rules explicitly rather than silently deleting unusual applicants.

Review DAYS_EMPLOYED=365243 as the known placeholder candidate. Negative birth and employment offsets are not inherently invalid.


In [30]:
financial_columns = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
]

range_report = pd.DataFrame({
    "minimum": df_clean[financial_columns].min(),
    "maximum": df_clean[financial_columns].max(),
    "zero_count": df_clean[financial_columns].eq(0).sum(),
    "negative_count": df_clean[financial_columns].lt(0).sum(),
    "missing_count": df_clean[financial_columns].isna().sum(),
})

display(range_report)

,minimum,maximum,zero_count,negative_count,missing_count
AMT_INCOME_TOTAL,25650.0,117000000.0,0,0,0
AMT_CREDIT,45000.0,4050000.0,0,0,0
AMT_ANNUITY,1615.5,258025.5,0,0,12
AMT_GOODS_PRICE,40500.0,4050000.0,0,0,278


In [31]:
# Convert birth-day offsets into approximate ages for inspection.
age_years = -df_clean["DAYS_BIRTH"] / 365.25

display(age_years.describe().round(2))

print(
    "Nonnegative birth offsets:",
    df_clean["DAYS_BIRTH"].ge(0).sum(),
)

print(
    "Employment placeholders:",
    df_clean["DAYS_EMPLOYED"].eq(365243).sum(),
)

print(
    "Other positive employment offsets:",
    (
        df_clean["DAYS_EMPLOYED"].gt(0)
        & df_clean["DAYS_EMPLOYED"].ne(365243)
    ).sum(),
)

count    307511.00
mean         43.91
std          11.95
min          20.50
25%          33.98
50%          43.12
75%          53.89
max          69.07
Name: DAYS_BIRTH, dtype: float64

Nonnegative birth offsets: 0
Employment placeholders: 55374
Other positive employment offsets: 0


## 8. Investigate outliers and document decisions
Compare median, upper percentiles, and maximum income, credit, and annuity. Display extreme records with related columns to assess consistency.

**Example:** our inspection found income of 117,000,000. Record it for review; its size alone does not establish a data error or justify dropping the application.

**Practice:** document retain, correct, flag, or pending review. If we later choose clipping or an IQR rule, calculate its thresholds using training rows only and reuse them for validation. Avoid choosing thresholds to improve validation results repeatedly.


In [32]:
outlier_summary = df_clean[financial_columns].describe(
    percentiles=[0.5, 0.95, 0.99]
).T

display(outlier_summary.round(2))

,count,mean,std,min,50%,95%,99%,max
AMT_INCOME_TOTAL,307511.0,168797.92,237123.15,25650.0,147150.0,337500.0,472500.0,117000000.0
AMT_CREDIT,307511.0,599026.00,402490.78,45000.0,513531.0,1350000.0,1854000.0,4050000.0
AMT_ANNUITY,307499.0,27108.57,14493.74,1615.5,24903.0,53325.0,70006.5,258025.5
AMT_GOODS_PRICE,307233.0,538396.21,369446.46,40500.0,450000.0,1305000.0,1800000.0,4050000.0


In [33]:
display(
    df_clean.nlargest(10, "AMT_INCOME_TOTAL")[
        [
            "SK_ID_CURR",
            "AMT_INCOME_TOTAL",
            "AMT_CREDIT",
            "AMT_ANNUITY",
            "NAME_INCOME_TYPE",
            "OCCUPATION_TYPE",
        ]
    ]
)

,SK_ID_CURR,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,NAME_INCOME_TYPE,OCCUPATION_TYPE
12840,114967,117000000.0,562491.0,26194.5,Working,Laborers
203693,336147,18000090.0,675000.0,69295.5,Commercial associate,NaN
246858,385674,13500000.0,1400503.5,130945.5,Commercial associate,NaN
77768,190160,9000000.0,1431531.0,132601.5,Working,Managers
131127,252084,6750000.0,790830.0,52978.5,Working,Laborers
103006,219563,4500000.0,2250000.0,225000.0,Working,Managers
187833,317748,4500000.0,835380.0,42651.0,Working,Laborers
204564,337151,4500000.0,450000.0,47749.5,Working,Managers
287463,432980,4500000.0,1755000.0,61132.5,Working,Managers
181698,310601,3950059.5,675000.0,66217.5,Commercial associate,Managers


## 9. Apply confirmed cleaning rules
Apply only rules supported by the investigations and update the decision log.

**First rule to implement together:** create DAYS_EMPLOYED_PLACEHOLDER (1 for 365243, 0 otherwise), then replace the exact value 365243 in DAYS_EMPLOYED with numeric missing. Our earlier inspection found 55,374 such rows: 55,352 pensioners and 22 unemployed applicants.

**Verify:** zero placeholders remain, the flag count matches the original placeholder count, missing employment values increase by that count, and row count is unchanged. Missing means duration unknown; it does not mean zero employment days.

Apply the same deterministic rules to future prediction data. Record the rules so they can be reproduced later. Do not fill values using full-data medians here.


In [37]:
# Identify placeholders using the unchanged original table.
# This preserves the flag when the cell is run again.
assert df_clean.index.equals(df.index), "Original and working rows must align."
employment_placeholder = df["DAYS_EMPLOYED"].eq(365243)

# Preserve the information that a placeholder was recorded.
df_clean["DAYS_EMPLOYED_PLACEHOLDER"] = employment_placeholder.astype(int)

# Replace the placeholder with a numeric missing value.
df_clean["DAYS_EMPLOYED"] = df_clean["DAYS_EMPLOYED"].replace(
    365243, float("nan")
)



print("Rows:", len(df_clean))

print(
    "Placeholders remaining:",
    df_clean["DAYS_EMPLOYED"].eq(365243).sum(),
)

print(
    "Flagged applications:",
    df_clean["DAYS_EMPLOYED_PLACEHOLDER"].sum(),
)

print(
    "Missing employment values:",
    df_clean["DAYS_EMPLOYED"].isna().sum(),
)


Rows: 307511
Placeholders remaining: 0
Flagged applications: 55374
Missing employment values: 55374


## 10. Separate inputs and outcome, then split
Store SK_ID_CURR separately; create y from TARGET and X from model inputs excluding TARGET and SK_ID_CURR. Keep the new employment flag in X.

Create an 80% training / 20% validation split using train_test_split, random_state=42, and stratify=y. Name the outputs X_train, X_valid, y_train, and y_valid.

**Verify:** matching X/y indexes, no overlap between split indexes, preserved total row count, and similar TARGET proportions. Retain split application IDs.

From this point onward, estimate replacements, feature-selection thresholds, category grouping, and scaling using training data only. Validation rows remain held aside for evaluation.


In [38]:
# Keep identifiers for matching predictions to applications.
application_ids = df_clean["SK_ID_CURR"].copy()

# The outcome we want to predict.
y = df_clean["TARGET"].copy()

# The information the model will learn from.
X = df_clean.drop(columns=["SK_ID_CURR", "TARGET"])

In [39]:
print("Input shape:", X.shape)
print("Target shape:", y.shape)

print("TARGET included in inputs:", "TARGET" in X.columns)
print("ID included in inputs:", "SK_ID_CURR" in X.columns)

print(
    "Employment flag included:",
    "DAYS_EMPLOYED_PLACEHOLDER" in X.columns,
)

print("Input and target indexes match:", X.index.equals(y.index))

Input shape: (307511, 121)
Target shape: (307511,)
TARGET included in inputs: False
ID included in inputs: False
Employment flag included: True
Input and target indexes match: True


In [ ]:


from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 3.1 MB/s eta 0:00:03
   ----- ---------------------------------- 1.0/8.4 MB 2.9 MB/s eta 0:00:03
   -------- ------------------------------- 1.8/8.4 MB 3.3 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.4 MB 2.8 MB/s eta 0:00:03
   --------------- ------------------------ 3.1/8.4 MB 3.3 MB/s eta 0:00:02
   ------------------ --------------------- 3.9/8.4 MB 3.2 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.4 MB 3.3 MB/s eta 0:00:02
   ------------------------- -------------- 5.2/8.4 MB 3.2 MB/s eta 0:00:01
   ---------------------------- ----------- 6.0/8.4 MB 3.3 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.4 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 3.8 MB/s  0:00:02
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
    ------------------------

In [42]:
print("Training inputs:", X_train.shape)
print("Validation inputs:", X_valid.shape)

print("Training labels aligned:", X_train.index.equals(y_train.index))
print("Validation labels aligned:", X_valid.index.equals(y_valid.index))

print(
    "Overlapping row indexes:",
    len(X_train.index.intersection(X_valid.index)),
)

display(pd.DataFrame({
    "training_percent": y_train.value_counts(normalize=True) * 100,
    "validation_percent": y_valid.value_counts(normalize=True) * 100,
}).round(2))

Training inputs: (246008, 121)
Validation inputs: (61503, 121)
Training labels aligned: True
Validation labels aligned: True
Overlapping row indexes: 0


,training_percent,validation_percent
TARGET,,
0,91.93,91.93
1,8.07,8.07


## 11. Finalize cleaning rules for application_train
Complete remaining whitespace and empty-string checks, validate labels, and apply only documented corrections to the labeled application data. Retain valid rare categories and keep XNA/Unknown as explicit categories for this first version unless evidence supports a different mapping. Record retained extreme incomes rather than claiming they were corrected.

**Practice:** define a reusable cleaning function accepting raw application rows and returning a cleaned copy. Include the exact employment-placeholder replacement and flag. Reconstruct the flag from raw values, or explicitly preserve an existing correct flag when supporting repeated cleaning. Never overwrite that flag with zeros after placeholders are gone.

Apply the finalized rules to df from application_train.csv. Preserve SK_ID_CURR, TARGET, indexes, and the original df. Update the decision log with actual changes and affected counts. Do not calculate full-data medians, learned cutoffs, or category-frequency thresholds here.


In [43]:
text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

text_checks = []

for column in text_columns:
    values = df_clean[column]
    stripped_values = values.str.strip()

    text_checks.append({
        "column": column,
        "values_with_outer_spaces": (
            values.notna() & values.ne(stripped_values)
        ).sum(),
        "empty_after_stripping": stripped_values.eq("").sum(),
    })

text_check_report = pd.DataFrame(text_checks)

display(text_check_report)

,column,values_with_outer_spaces,empty_after_stripping
0,NAME_CONTRACT_TYPE,0,0
1,CODE_GENDER,0,0
2,FLAG_OWN_CAR,0,0
3,FLAG_OWN_REALTY,0,0
4,NAME_TYPE_SUITE,0,0
5,NAME_INCOME_TYPE,0,0
6,NAME_EDUCATION_TYPE,0,0
7,NAME_FAMILY_STATUS,0,0
8,NAME_HOUSING_TYPE,0,0
9,OCCUPATION_TYPE,0,0


In [44]:
def clean_applications(raw_data):
    # This function expects the original, uncleaned application table.
    if "DAYS_EMPLOYED_PLACEHOLDER" in raw_data.columns:
        raise ValueError("Pass the original data, df, to this function.")

    cleaned = raw_data.copy()

    # Standardize outer spaces and turn empty text into missing.
    text_columns = cleaned.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        cleaned[column] = (
            cleaned[column]
            .str.strip()
            .replace("", float("nan"))
        )

    # Preserve the employment-placeholder information.
    placeholder = cleaned["DAYS_EMPLOYED"].eq(365243)

    cleaned["DAYS_EMPLOYED_PLACEHOLDER"] = placeholder.astype(int)

    cleaned["DAYS_EMPLOYED"] = cleaned["DAYS_EMPLOYED"].replace(
        365243, float("nan")
    )

    return cleaned

df_clean = clean_applications(df)

In [ ]:
print("Cleaned shape:", df_clean.shape)

print(
    "Employment flags:",
    df_clean["DAYS_EMPLOYED_PLACEHOLDER"].sum(),
)

print(
    "Placeholders remaining:",
    df_clean["DAYS_EMPLOYED"].eq(365243).sum(),
)

print(
    "Original placeholders still present:",
    df["DAYS_EMPLOYED"].eq(365243).sum(),
)

Cleaned shape: (307511, 123)
Employment flags: 55374
Placeholders remaining: 0
Original placeholders still present: 55374


: 

## 12. Engineer features from the labeled application table
Create useful model inputs using only the current application's recorded information. Suggested candidates: approximate age (-DAYS_BIRTH / 365.25), observed employment duration after placeholder handling, credit-to-income, annuity-to-income, credit-to-goods-price, and an OWN_CAR_AGE_MISSING indicator. Retain the employment-placeholder flag and car ownership.

**Practice:** implement a reusable feature function with documented names, formulas, units, and input requirements. A loan of 300,000 and income of 150,000 gives credit-to-income ratio 2. Do not interpret annuity-to-income as a monthly burden unless time units are verified.

Set ratios with zero, missing, or nonfinite denominators to missing; never leave infinities. Unknown employment duration stays missing. Verify row count/index preservation. Document whether original day offsets are retained or removed to avoid unexplained duplicate representations. Do not use TARGET or information from future outcomes.

This section uses application_train.csv only. We will process the other CSVs later.


## 13. Rebuild the training and validation inputs consistently
Section 10 already established an 80/20 split from application_train.csv. Preserve its application membership using the saved training and validation indexes; keep the labels aligned.

**Practice:** reconstruct X from the finalized cleaned/engineered labeled table, excluding SK_ID_CURR and TARGET. Rebuild X_train and X_valid using the existing split indexes; preserve y_train/y_valid and application IDs in the matching row order. Do not keep using stale feature tables created before the new cleaning or features.

Verify disjoint indexes, no lost/duplicated rows, matching X/y indexes, and similar target proportions. The validation rows are part of application_train.csv; the separate Kaggle application_test.csv is not used in this notebook's current work.


## 14. Choose retained features and missing-data strategies
Review X_train for missing percentages, all-empty features, constant columns, and semantic usefulness. Document decisions to retain, exclude, fill, or flag. Keep IDs and TARGET out of model features.

**Starting strategy:** retain useful partly observed features, use training medians for numeric gaps, and use an explicit missing label for text. Handle all-empty training columns explicitly; a median cannot be estimated without recorded training values. Do not automatically remove every feature above 20% missing.

**Practice:** save the retained feature names and order, applying exactly that selection to X_train and X_valid. Any missingness cutoff, constant-feature selection, rare-category grouping, or outlier threshold must be learned from training rows only. Review validation independently for quality but do not use its statistics to fit rules.


## 15. Identify numeric and categorical feature groups
Inspect the retained X_train feature types and create numeric and categorical feature lists. pandas may report text as object, string, or str; include actual observed text types.

**Practice:** verify every retained feature belongs to exactly one branch, no features are silently omitted, and validation inputs use the same expected schema. Resolve confirmed mixed-type problems with explicit documented rules. Numeric binary flags can remain numeric.

Check that TARGET and SK_ID_CURR are absent, expected engineered features are present, and placeholder indicators survived the rebuild.


## 16. Define numeric missing-value handling
Define a numeric preprocessing branch using SimpleImputer with median strategy, after explicitly resolving all-empty training features.

**Example:** recorded training values 10, 20, and 1000 have median 20. That training-derived replacement is reused for both training and validation gaps.

**Practice:** define the branch without fitting it. Decide whether additional missingness indicators are useful or duplicate existing explicit flags. Do not fill the entire labeled CSV using a median calculated before splitting. Check the installed scikit-learn API when implementing the branch.


## 17. Define text missing-value handling and encoding
Define a categorical branch that fills actual missing entries with a reserved explicit label, then uses OneHotEncoder(handle_unknown="ignore") with sparse output. Confirm the missing label is not already a legitimate category and check missing-value compatibility with the installed pandas/scikit-learn versions.

**Example:** NAME_CONTRACT_TYPE becomes Cash loans and Revolving loans indicator columns. XNA remains an explicit recorded category. Do not guess gender, organization, or other unavailable information.

**Practice:** define this branch without fitting it. Categories must be learned from X_train only. Validation labels not seen during fitting must not create a different feature layout; document how the encoder represents them.


## 18. Decide scaling and assemble reusable preprocessing
Scaling changes feature magnitudes and is model-dependent. Agree on the expected first model with your teammate before finalizing export.

**Practice:** combine numeric and categorical branches with ColumnTransformer. If the first model is scale-sensitive, put a scaler after numeric imputation inside the numeric branch; otherwise produce an unscaled prepared baseline and record the choice. Tree models generally do not need scaling; logistic regression generally benefits from it.

Fit scaling only on training rows. Set feature lists and remainder behavior explicitly. Avoid centering a combined sparse matrix or unnecessarily converting it to a large dense table. Preserve an unscaled option or raw inputs so the teammate can compare other model families.


## 19. Fit preprocessing on training and transform validation
Use fit_transform on X_train and transform on X_valid with the same preprocessor. The validation portion remains independent of all fitted replacements, category vocabularies, scaling, and learned selection thresholds.

**Practice:** retain the matrices, transformed feature names, and matching labels/IDs in row order. Do not call fit or fit_transform on validation rows.

The source file is still application_train.csv for both portions. We do not generate a single full-data encoded table by independently learning categories and replacements from all its rows. That would undermine validation.

For cross-validation, your teammate must refit preprocessing within each training fold using the reusable code and unimputed inputs we supply.


## 20. Verify the cleaned and prepared labeled data
Compare raw, cleaned, engineered, and transformed stages. Verify row counts, IDs, outcomes, employment flags, valid ratios, feature order, split membership, and X/y alignment. Explain every intentional column-count change.

**Practice:** confirm no IDs or TARGET entered features, no split overlap, and equal transformed feature counts/order for training and validation. Check prepared numeric values for missingness and infinity without densifying a sparse matrix. Inspect unseen validation categories and their handling.

Verify all fitted statistics came from training only. Missing values may increase after replacing invalid placeholders before imputation; distinguish intermediate from final checks. These checks establish input integrity, not predictive accuracy.


## 21. Finish the cleaning and preprocessing report
Record raw/cleaned/prepared dimensions, missing-value findings, categorical decisions, outlier decisions, corrections and affected counts, engineered formulas, exclusions, imputation, encoding, scaling choice, and verification results.

**Practice:** mark decision-log actions as applied only after execution and verification. Include split settings, target proportions, dependency versions, unresolved assumptions, and a clear statement that this handoff uses application_train.csv only. No Kaggle test or historical CSV preprocessing has been completed in this notebook's current scope.

Explain what your teammate receives and which work remains: model training, tuning, prediction-quality evaluation, and later extension to other CSVs.


## 22. Save and reload the application_train handoff
After verification, save artifacts for the labeled application data only:

- The cleaned labeled table, retaining IDs and TARGET, with its pre/post-feature-engineering stage clearly named.
- Prepared training and validation matrices in an appropriate sparse or dense format, matching labels, ordered application IDs, and transformed feature names.
- Unimputed input splits or a reproducible way to reconstruct them, plus the fitted preprocessor, retained raw feature names, deterministic cleaning/feature code, and configuration.
- Split membership/settings, the decision log, verification report, and dependency versions.

**Practice:** create outputs only when this section is implemented. Reload them and confirm shapes, IDs, feature order, and an example transformation. Keep raw CSVs intact. The fitted preprocessing supports the initial held-out evaluation; the raw split data and unfitted code support fold-specific preprocessing during cross-validation.

**Completion point:** your teammate can start ML training on prepared application-table data. We will then process application_test.csv and historical CSVs as separately tracked work, reusing rules where appropriate. Neither is required to train the first labeled-data baseline, but test preprocessing is required before predicting those applications.
